# Exploratory C-MAPSS RUL baseline

Train a small, reproducible regression baseline from the processed C-MAPSS train and validation splits. This is an exploratory benchmark for the project pipeline, not a production evaluation. The labels below use each observed unit's last cycle and should not be interpreted as an online deployment metric.

The processed Parquet files are tracked with DVC. Run `dvc pull` from the repository root if they are missing.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent
processed = ROOT / "data" / "processed"
train_path = processed / "cmapss_train.parquet"
val_path = processed / "cmapss_val.parquet"
missing = [str(path) for path in (train_path, val_path) if not path.exists()]
if missing:
    raise FileNotFoundError("Missing processed split(s): " + ", ".join(missing) + ". Run `dvc pull` from the repository root.")

train_long = pd.read_parquet(train_path)
val_long = pd.read_parquet(val_path)
train_long.head()


In [ ]:
def make_features(long_frame):
    frame = long_frame.copy()
    frame["cycle"] = ((pd.to_datetime(frame["time"]) - pd.Timestamp("2025-01-01")) / pd.Timedelta(hours=1)).round().astype(int)
    wide = frame.pivot_table(index=["machine_id", "cycle"], columns="sensor", values="value", aggfunc="mean")
    wide.columns = [f"sensor_{column}" for column in wide.columns]
    wide = wide.reset_index()
    max_cycle = wide.groupby("machine_id")["cycle"].transform("max")
    wide["rul_cycles"] = max_cycle - wide["cycle"]
    return wide

train = make_features(train_long)
valid = make_features(val_long)
feature_columns = [column for column in train.columns if column.startswith("sensor_")]
print(f"Train rows: {len(train):,}; validation rows: {len(valid):,}; features: {len(feature_columns)}")
train[feature_columns + ["rul_cycles"]].head()


In [ ]:
model = HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=15, random_state=42)
model.fit(train[feature_columns], train["rul_cycles"])
val_predictions = model.predict(valid[feature_columns])
rmse = np.sqrt(mean_squared_error(valid["rul_cycles"], val_predictions))
print(f"Validation RMSE: {rmse:.2f} cycles")
